# Sigma Sweep v2 -- Gap Decomposition (corrected symmetric scorer)

Rebuilds the paper's main decomposition table. The original sweep (git 4ca8e88) used
an **asymmetric scorer** (members never queried the target model) -- every ε_lower in
the old findings.md table is untrustworthy.

**v2 design:** corrected scorer imported from `run_raw_lira_pilot.py` (not reimplemented);
full MNIST, 1 epoch, σ ∈ {0.5,...,5.0}, K=32 shadows/σ (inheriting the target's σ -- old
helper hardcoded 1.1); budget 2048/seed (pooled ~5120/5120) with disjoint per-seed draws;
three estimators per cell (Wilson-conservative headline, point, GDP diagnostic-only);
validity gate + `censored` flag. Results persist after every σ, so a crash loses nothing.

Expected: the σ=0.5 accounting share (~93%) survives; σ≈2.0 shares are sensitive and are
the reason for this rerun.

Upload `sigma_sweep_bundle_v2.zip` when prompted.

In [ ]:
from google.colab import files
up = files.upload()  # choose sigma_sweep_bundle_v2.zip
import zipfile
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
print("Extracted", zname)

In [ ]:
%pip -q install opacus dp-accounting pyyaml scipy
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY -- ABORT: runtime will be ~10-20x slower; Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- PRE-FLIGHT SANITY CHECKS (abort before any training) -------------------
# Embedded from research/validate_estimator_scorer_20260705.py (the bundle has
# no tests/). Verifies the estimator core + corrected scorer logic and that the
# bundle actually contains the 2026-07-05 fixes. Aborts the notebook if any fail.
import sys, random, statistics
sys.path.insert(0, "src"); sys.path.insert(0, "codex")
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
import run_raw_lira_pilot as pilot
import inspect

random.seed(608)
kw = dict(delta=1e-5, align_event_to_score_direction=True,
          require_member_favoring=True, report_confidence_supported_lower_bound=True)
def est(m, n, direction="higher"):
    return estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                          score_direction=direction, **kw).epsilon_lower_empirical
checks = []
m=[random.gauss(0,1) for _ in range(640)]; n=[random.gauss(0,1) for _ in range(640)]
checks.append(("T1 no-signal -> eps=0", est(m,n) < 0.05))
m=[random.gauss(3,1) for _ in range(640)]; n=[random.gauss(0,1) for _ in range(640)]
checks.append(("T2a signal+higher -> eps>0", est(m,n,"higher") > 0.5))
checks.append(("T2b signal+lower -> eps=0", est(m,n,"lower") < 0.05))
K,B=32,640
mem_a,non_a,mem_s,non_s=[],[],[],[]
for _ in range(B):
    sl=[random.gauss(2.0,0.5) for _ in range(K)]; t=random.gauss(2.0,0.5)
    mem_a.append(statistics.fmean(sl[K//2:])-statistics.fmean(sl[:K//2]))
    mem_s.append(statistics.fmean(sl[K//2:])-t)
for _ in range(B):
    sl=[random.gauss(2.0,0.5) for _ in range(K)]; t=random.gauss(2.0,0.5)
    non_a.append(statistics.fmean(sl)-t); non_s.append(statistics.fmean(sl)-t)
checks.append(("T3a buggy asymmetric scorer, no signal -> spurious eps",
               max(est(mem_a,non_a,"higher"), est(mem_a,non_a,"lower")) > 0.2))
checks.append(("T3b fixed symmetric scorer, no signal -> eps~0", est(mem_s,non_s,"higher") < 0.05))
mem2=[statistics.fmean([random.gauss(2.0,0.5) for _ in range(K//2)])-random.gauss(0.8,0.4) for _ in range(B)]
non2=[statistics.fmean([random.gauss(2.0,0.5) for _ in range(K)])-random.gauss(2.0,0.5) for _ in range(B)]
e4=est(mem2,non2,"higher")
checks.append(("T4 symmetric scorer + real signal -> eps>0", e4 > 0.5))
checks.append(("T5 validity-gate comparison well-defined", (e4 > 0.771) in (True, False)))
# bundle fix markers
src = inspect.getsource(pilot.raw_lira_scores)
checks.append(("bundle has SYMMETRIC scorer", "fmean(out_losses) - float(target_" in src.replace("\n","")))
checks.append(("bundle has disjoint sampler", hasattr(pilot, "sample_query_indices_disjoint")))
checks.append(("bundle has quality flags", hasattr(pilot, "apply_quality_flags")))
failed = [name for name, ok in checks if not ok]
for name, ok in checks: print(("PASS " if ok else "FAIL ") + name)
if failed:
    raise SystemExit(f"SANITY CHECKS FAILED: {failed} -- do NOT run the experiment.")
print("\nAll sanity checks passed -- safe to run.")

In [ ]:
!python codex/run_sigma_sweep.py

In [ ]:
import json, pandas as pd
rows = json.load(open("codex/results/sigma_sweep_v2/sigma_sweep_summary.json"))
df = pd.DataFrame(rows)
cols = ["sigma","epsilon_upper_rdp","epsilon_upper_tighter","epsilon_lower_conservative",
        "epsilon_lower_point","epsilon_lower_gdp","accuracy","validity","censored"]
display(df[cols].sort_values("sigma"))
print(open("codex/results/sigma_sweep_v2/sigma_sweep_decomposition.md").read())

In [ ]:
import matplotlib.pyplot as plt
ok = df[(df.validity == "ok")]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ok.sigma, ok.epsilon_upper_rdp, "o-", label="eps_upper RDP")
ax[0].plot(ok.sigma, ok.epsilon_upper_tighter, "s-", label="eps_upper PLD")
ax[0].plot(ok.sigma, ok.epsilon_lower_conservative, "^-", label="eps_lower (Wilson)")
ax[0].set_yscale("log"); ax[0].set_xlabel("sigma"); ax[0].legend(); ax[0].set_title("Bounds vs sigma")
acct = (ok.epsilon_upper_rdp - ok.epsilon_upper_tighter) / (ok.epsilon_upper_rdp - ok.epsilon_lower_conservative) * 100
ax[1].plot(ok.sigma, acct, "o-")
ax[1].set_xlabel("sigma"); ax[1].set_ylabel("accounting share of total gap (%)"); ax[1].set_title("Gap decomposition")
plt.tight_layout(); plt.show()

In [ ]:
!cd codex/results && zip -r ../../sigma_sweep_results_v2.zip sigma_sweep_v2
from google.colab import files
files.download("sigma_sweep_results_v2.zip")